# 06_Scikit_Learn.ipynb

# 1. Introduction

## What is ElasticNet in Scikit-Learn?

`ElasticNet` is Scikit-Learn's implementation of **Elastic Net Regression**, a linear regression algorithm that combines **L1 (Lasso)** and **L2 (Ridge)** regularization.

It is used when you want to:

* Reduce overfitting
* Handle multicollinearity
* Perform automatic feature selection
* Build more stable linear models

Internally, Scikit-Learn's `ElasticNet` uses the **Coordinate Descent** optimization algorithm.

---

## Import

```python
from sklearn.linear_model import ElasticNet
```

---

## Basic Workflow

```python
from sklearn.linear_model import ElasticNet

model = ElasticNet()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
```

---

## When should you use ElasticNet?

Use ElasticNet when:

* Features are highly correlated.
* You want feature selection.
* Ridge keeps too many features.
* Lasso removes too many correlated features.
* You need a balance between Ridge and Lasso.

---

# 2. Import & Constructor

## Required Imports

```python
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
```

---

## Constructor Syntax

```python
ElasticNet(
    alpha=1.0,
    l1_ratio=0.5,
    fit_intercept=True,
    max_iter=1000,
    tol=1e-4,
    warm_start=False,
    positive=False,
    random_state=None,
    selection="cyclic"
)
```

---

## Parameter Table

| Parameter       | Default    | Description                           | Common Usage                                        |
| --------------- | ---------- | ------------------------------------- | --------------------------------------------------- |
| `alpha`         | `1.0`      | Overall regularization strength       | Tune using GridSearchCV                             |
| `l1_ratio`      | `0.5`      | Balance between L1 and L2 penalties   | Tune between 0 and 1                                |
| `fit_intercept` | `True`     | Learn intercept (`b`)                 | Keep `True` unless data is centered                 |
| `max_iter`      | `1000`     | Maximum Coordinate Descent iterations | Increase if convergence warning appears             |
| `tol`           | `1e-4`     | Stopping tolerance                    | Leave default in most cases                         |
| `warm_start`    | `False`    | Reuse previous coefficients           | Useful for repeated fitting                         |
| `positive`      | `False`    | Force coefficients to be positive     | Rarely used, only when required by domain knowledge |
| `selection`     | `"cyclic"` | Order of coefficient updates          | `"cyclic"` is recommended                           |

---

## Recommended Settings

For most datasets:

```python
model = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    max_iter=5000,
    random_state=42
)
```

---

## Best Practices

* Always scale numerical features.
* Tune `alpha` and `l1_ratio` instead of using defaults.
* Increase `max_iter` if training doesn't converge.
* Use `random_state` when `selection="random"` for reproducibility.
* Save the scaler together with the trained model.

---

# 3. Methods & Attributes

## Methods Table

| Method         | Purpose                    | Returns           |
| -------------- | -------------------------- | ----------------- |
| `fit(X, y)`    | Train the model            | Trained estimator |
| `predict(X)`   | Predict target values      | NumPy array       |
| `score(X, y)`  | Compute R² score           | Float             |
| `get_params()` | Get constructor parameters | Dictionary        |
| `set_params()` | Update parameters          | Estimator         |

---

## Attributes Table

| Attribute           | Description                       |
| ------------------- | --------------------------------- |
| `coef_`             | Learned feature coefficients      |
| `intercept_`        | Learned intercept                 |
| `n_iter_`           | Number of iterations used         |
| `feature_names_in_` | Feature names (if DataFrame used) |
| `n_features_in_`    | Number of input features          |

---

## Complete Example

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

# Dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model
model = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    random_state=42
)

# Training
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Commonly Used Attributes
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("Iterations:", model.n_iter_)

# Commonly Used Method
print("R²:", model.score(X_test, y_test))
```

---

## Commonly Used Methods

### `fit()`

Learns the coefficients from the training data.

```python
model.fit(X_train, y_train)
```

---

### `predict()`

Predicts values for unseen data.

```python
y_pred = model.predict(X_test)
```

---

### `score()`

Returns the coefficient of determination (**R²**).

```python
model.score(X_test, y_test)
```

---

## Commonly Used Attributes

### `coef_`

```python
model.coef_
```

Returns the learned coefficients.

Features with coefficient **0** have been removed due to the L1 penalty.

---

### `intercept_`

```python
model.intercept_
```

Returns the learned bias term.

---

### `n_iter_`

```python
model.n_iter_
```

Shows how many Coordinate Descent iterations were required for convergence.

---

# 4. End-to-End Workflow

## Workflow Diagram

```text
Load Data
    ↓
Train-Test Split
    ↓
StandardScaler
    ↓
Create ElasticNet Model
    ↓
Train Model
    ↓
Predict
    ↓
Evaluate
```

---
---

## Important Notes

* Always standardize features before training.
* ElasticNet is sensitive to feature scales because of regularization.
* `alpha` controls the regularization strength.
* `l1_ratio` controls the mix of Ridge and Lasso.
* Coordinate Descent is used internally, so increasing `max_iter` may be necessary for difficult datasets.

---

## Common Errors

| Mistake                                | Why It Happens                                            |
| -------------------------------------- | --------------------------------------------------------- |
| Not scaling features                   | Regularization becomes biased toward large-scale features |
| Very large `alpha`                     | Model underfits due to excessive regularization           |
| Very small `alpha`                     | Model behaves like Linear Regression and may overfit      |
| Ignoring `ConvergenceWarning`          | `max_iter` is too small or `tol` is too strict            |
| Forgetting to transform test data      | Model expects scaled inputs during prediction             |
| Calling `fit_transform()` on test data | Causes data leakage; use `transform()` instead            |

---

In [5]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing(as_frame=True)

X = housing.data
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.1,
        l1_ratio=0.5,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("R²:", pipeline.score(X_test, y_test))

R²: 0.5147647043408876
